# 02a - Encoder Smoke Test

This notebook does only two things:
- install the package stack needed by the official Concerto backend
- run a smoke test for `src/encoder.py` on a dummy point cloud

If this passes, the next step is `notebooks/02_feature_extraction.ipynb`.


### 1. Mount Drive and Get the Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
REPO_URL = 'https://github.com/Gandata/Deep_learning_project.git'
BRANCH = 'dev/enc-mlp'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull origin {BRANCH}
!git branch --show-current
!git log -1 --oneline


### 2. Install Project + Concerto Stack

In [ ]:
import os
from pathlib import Path
import re
import subprocess
import sys

PINNED_TORCH = '2.5.0'
PINNED_TORCHVISION = '0.20.0'
PINNED_TORCHAUDIO = '2.5.0'
PINNED_TORCH_CUDA_TAG = '124'
CONCERTO_DIR = '/content/Concerto'
os.environ['CONCERTO_DIR'] = CONCERTO_DIR

def run(cmd: str, check: bool = True, capture: bool = False, cwd: str | None = None):
    print(f'+ {cmd}')
    completed = subprocess.run(
        cmd,
        shell=True,
        check=False,
        text=True,
        capture_output=capture,
        cwd=cwd,
    )
    if capture and completed.stdout:
        print(completed.stdout.strip())
    if capture and completed.stderr:
        print(completed.stderr.strip())
    if check and completed.returncode != 0:
        raise RuntimeError(
            f'Command failed with exit code {completed.returncode}: {cmd}'
        )
    return completed

def detect_cuda_tag_from_nvcc() -> str | None:
    completed = run('nvcc --version', check=False, capture=True)
    if completed.returncode != 0:
        return None
    match = re.search(r'release\\s+(\\d+)\\.(\\d+)', completed.stdout + completed.stderr)
    if match is None:
        return None
    return f"{match.group(1)}{match.group(2)}"

run('pip install -q -r requirements.txt')
run('python --version')
run('pip uninstall -y torch torchvision torchaudio spconv spconv-cu118 spconv-cu120 spconv-cu121 spconv-cu124 spconv-cu125 spconv-cu126 spconv-cu128 cumm cumm-cu118 cumm-cu120 cumm-cu121 cumm-cu124 cumm-cu125 cumm-cu126 cumm-cu128 torch-scatter || true')
run(
    f'pip install --no-cache-dir torch=={PINNED_TORCH} torchvision=={PINNED_TORCHVISION} torchaudio=={PINNED_TORCHAUDIO} --index-url https://download.pytorch.org/whl/cu{PINNED_TORCH_CUDA_TAG}'
)

import torch

if torch.version.cuda is None:
    raise RuntimeError(
        'CUDA-enabled PyTorch was not installed. Restart the runtime on a T4 GPU and rerun from the top.'
    )

torch_version = torch.__version__.split('+')[0]
torch_cuda_tag = torch.version.cuda.replace('.', '')
nvcc_cuda_tag = detect_cuda_tag_from_nvcc()

print('torch version :', torch.__version__)
print('torch cuda tag:', torch_cuda_tag)
print('nvcc cuda tag :', nvcc_cuda_tag)

if torch_version != PINNED_TORCH or torch_cuda_tag != PINNED_TORCH_CUDA_TAG:
    raise RuntimeError(
        f'Expected torch {PINNED_TORCH}+cu{PINNED_TORCH_CUDA_TAG}, got {torch.__version__}.'
    )

if not os.path.exists(CONCERTO_DIR):
    run(f'git clone https://github.com/Pointcept/Concerto.git {CONCERTO_DIR}')
else:
    run('git pull', cwd=CONCERTO_DIR)

spconv_candidates = []
for candidate in [nvcc_cuda_tag, torch_cuda_tag, '126', '125', '124', '121', '120', '118']:
    if candidate and candidate not in spconv_candidates:
        spconv_candidates.append(candidate)

spconv_ok = False
last_spconv_error = None
for spconv_cuda_tag in spconv_candidates:
    run('pip uninstall -y spconv spconv-cu118 spconv-cu120 spconv-cu121 spconv-cu124 spconv-cu125 spconv-cu126 spconv-cu128 cumm cumm-cu118 cumm-cu120 cumm-cu121 cumm-cu124 cumm-cu125 cumm-cu126 cumm-cu128 || true')
    install_result = run(
        f'pip install --no-cache-dir spconv-cu{spconv_cuda_tag}',
        check=False,
        capture=True,
    )
    if install_result.returncode != 0:
        last_spconv_error = install_result.stderr or install_result.stdout
        print(f'spconv-cu{spconv_cuda_tag} install failed, trying next candidate...')
        continue

    import_result = run(
        'python -c "import spconv.pytorch as spconv; print(\'spconv import ok:\', spconv.__file__)"',
        check=False,
        capture=True,
    )
    if import_result.returncode == 0:
        spconv_ok = True
        print(f'Selected spconv-cu{spconv_cuda_tag}.')
        break

    last_spconv_error = import_result.stderr or import_result.stdout
    print(f'spconv-cu{spconv_cuda_tag} import failed, trying next candidate...')

if not spconv_ok:
    print(
        'Wheel-based spconv failed on this Colab image. Falling back to source builds '
        'for cumm/spconv at matching upstream tags.'
    )
    cumm_tag = 'v0.7.11'
    spconv_tag = 'v2.3.8'
    cumm_dir = f"/content/cumm-{cumm_tag.lstrip('v')}"
    spconv_src_dir = f"/content/spconv-{spconv_tag.lstrip('v')}"

    os.environ['CUMM_CUDA_ARCH_LIST'] = '7.5'
    os.environ['MAX_JOBS'] = '2'
    os.environ['CUDA_HOME'] = '/usr/local/cuda'

    run('pip uninstall -y spconv spconv-cu118 spconv-cu120 spconv-cu121 spconv-cu124 spconv-cu125 spconv-cu126 spconv-cu128 cumm cumm-cu118 cumm-cu120 cumm-cu121 cumm-cu124 cumm-cu125 cumm-cu126 cumm-cu128 || true')
    run('pip install --no-cache-dir pccm ccimport pybind11 ninja setuptools wheel')

    if not os.path.exists(cumm_dir):
        run(f'git clone --branch {cumm_tag} --depth 1 https://github.com/FindDefinition/cumm.git {cumm_dir}')
    if not os.path.exists(spconv_src_dir):
        run(f'git clone --branch {spconv_tag} --depth 1 https://github.com/traveller59/spconv.git {spconv_src_dir}')

    run('pip install -e . --no-build-isolation --no-deps', cwd=cumm_dir)
    run('python -c "import cumm; print(\'cumm import ok:\', cumm.__file__)"')

    spconv_pyproject = Path(spconv_src_dir) / 'pyproject.toml'
    pyproject_text = spconv_pyproject.read_text(encoding='utf-8')
    pyproject_text = pyproject_text.replace(', "cumm >= 0.7.11, < 0.8.0"', '')
    pyproject_text = pyproject_text.replace('"cumm >= 0.7.11, < 0.8.0", ', '')
    spconv_pyproject.write_text(pyproject_text, encoding='utf-8')

    run('pip install -e . --no-build-isolation --no-deps', cwd=spconv_src_dir)
    source_import_result = run(
        'python -c "import spconv.pytorch as spconv; print(\'spconv import ok:\', spconv.__file__)"',
        check=False,
        capture=True,
    )
    if source_import_result.returncode != 0:
        raise RuntimeError(
            'Both the wheel path and the source-build fallback failed for spconv. '
            f'Last source-build error:\\n{source_import_result.stderr or source_import_result.stdout or last_spconv_error}'
        )

run(
    f'pip install torch-scatter -f https://data.pyg.org/whl/torch-{PINNED_TORCH}+cu{PINNED_TORCH_CUDA_TAG}.html'
)
run(
    'python -c "import torch_scatter; print(\'torch_scatter import ok:\', torch_scatter.__file__)"'
)

run('pip install huggingface_hub timm')
run('pip install -e .', cwd=CONCERTO_DIR)
run('python -c "import concerto; print(\'concerto import ok:\', concerto.__file__)"')


### 3. Optional Hugging Face Token or Local Checkpoint

If `Pointcept/Concerto` downloads work anonymously, you can leave both options empty.
If not, either provide `HF_TOKEN` through Colab secrets or set a local `CHECKPOINT_PATH`.


In [ ]:
from pathlib import Path
import os

CHECKPOINT_PATH = None
# Example:
# CHECKPOINT_PATH = '/content/drive/MyDrive/DL_Project/pretrained/concerto_small.pth'

HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = None

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    Path('.env').write_text(f'HF_TOKEN={HF_TOKEN}\n', encoding='utf-8')
    print('Loaded HF_TOKEN from Colab secrets.')
else:
    print('No HF_TOKEN found in Colab secrets.')
    print('If the Hugging Face download is public, the smoke test may still work.')
    print('Otherwise set CHECKPOINT_PATH before running the next cell.')

if CHECKPOINT_PATH is not None:
    assert Path(CHECKPOINT_PATH).exists(), f'Checkpoint not found: {CHECKPOINT_PATH}'
    print('Using local checkpoint:', CHECKPOINT_PATH)


### 4. Smoke Test `src/encoder.py`

In [ ]:
import numpy as np
import torch

from src.encoder import ConcertoEncoder

device = 'cuda' if torch.cuda.is_available() else 'cpu'
checkpoint_path = CHECKPOINT_PATH if CHECKPOINT_PATH else None

print('device         :', device)
print('checkpoint path:', checkpoint_path)

encoder = ConcertoEncoder(
    checkpoint_path=checkpoint_path,
    device=device,
)

num_points = 2048
coord = np.random.randn(num_points, 3).astype(np.float32)
color = (np.random.rand(num_points, 3) * 255.0).astype(np.float32)

encoder_input = {
    'coord': coord,
    'color': color,
}

with torch.no_grad():
    features = encoder(encoder_input)

print('feature shape :', tuple(features.shape))
print('feature dtype :', features.dtype)
print('all finite    :', torch.isfinite(features).all().item())

assert tuple(features.shape) == (num_points, 256)
assert torch.isfinite(features).all().item()
print('Encoder smoke test passed.')
